### 크롤링에 필요한 라이브러리 설치

In [ ]:
# # 정적페이지 크롤링
# %conda install requests beautifulsoup4 lxml

# # 동적페이지 크롤링
# %conda install selenium

# # 동적페이지 크롤링 - ChromeDriver 다운로드
# %conda install webdriver-manager

# # 데이터 처리
# %conda install pandas

%conda install requests beautifulsoup4 lxml selenium pandas

### 1. 정적 크롤링의 큰 흐름

In [ ]:
import requests
from bs4 import BeautifulSoup

# 1. http 요청
url = "http://example.com/"
response = requests.get(url)

# 2. html 파싱 : 특정 데이터에서 원하는 데이터만 추출
soup = BeautifulSoup(response.text, 'lxml')

# 3. 데이터 추출
title = soup.find('h1').text
print(title)


### ** html 샘플 데이터

In [ ]:
from bs4 import BeautifulSoup

html = """
<html>
  <head><title>샘플 페이지</title></head>
  <body>
    <h1 class="main-title">제목</h1>
    <div id="content">
      <p class="text">첫 번째 문단</p>
      <p class="text">두 번째 문단</p>
      <a href="/page1">링크1</a>
      <a href="/page2">링크2</a>
    </div>
  </body>
</html>
"""

# soup = BeautifulSoup(html, 'lxml')
soup = BeautifulSoup(html, 'html.parser')

### 2. 데이터 찾기 메소드 (파싱 메소드)

In [ ]:
# 1) 태그로 찾기
title = soup.find('h1')
print(title.text)

for p in soup.find_all('p'):
    print(p.text)

# find : 가장 먼저 걸리는거 하나만 반환
# find all : 해당 내용 전부 다 리스트로 반환


# 2) css 선택자 (select)
# class 로 찾기
texts = soup.select('.text')

# id로 찾기
contents = soup.select('#content')

# 복합 선택자
links = soup.select('div#content a')

# 태그명으로 찾기
soup.select('p') 

### 3. 태그 안 속성값 가져오기

In [ ]:
# href 속성 가져오기
link = soup.find('a')
print(link['href'])
print(link.get('href')) # 안전

# 모든 링크 추출
for p in soup.select('a'):
    print(p.get('href'))


### 4. 실제 사이트 정적 크롤링
https://books.toscrape.com/

In [ ]:
import requests
from bs4 import BeautifulSoup
url = "https://books.toscrape.com/"
response = requests.get(url)
# response.raise_for_status() # 에러체크

soup = BeautifulSoup(response.text, 'html.parser')
# 아티클에 있는 속성 중에 이름이 product_pod 인 것들
soup.select('article.product_pod')[0].find('h3').text
soup.select('article.product_pod')[0].find('h3').find('a').get('title')
soup.select('article.product_pod')[0].select_one('h3 a').get('title')

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
url = "https://books.toscrape.com/"
response = requests.get(url)
# response.raise_for_status() # 에러체크

soup = BeautifulSoup(response.text, 'html.parser')
titles = []
links = []

for li in soup.select('ol.row li '):
    # print(li.select('article.product_pod')[0].select_one('h3 a').get('title'))
    titles.append(li.select('article.product_pod')[0].select_one('h3 a').get('title'))
    links.append(li.select('article.product_pod')[0].select_one('h3 a').get('href'))

# 시각화======================
data = {
    'title' : titles,
    'links' : links
}
pd.DataFrame(data)

In [46]:
# 페이지 모두 돌면서 정보 가져오기
import pandas as pd
import requests
from bs4 import BeautifulSoup
titles, links = [], []

for pagenum in range(1,51):
    url = f'https://books.toscrape.com/catalogue/page-{pagenum}.html'
    response = requests.get(url)
    response.raise_for_status()  # 에러체크

    soup = BeautifulSoup(response.text, 'lxml')

    for li in soup.select('ol.row li '):
        # print(li.select('article.product_pod')[0].select_one('h3 a').get('title'))
        titles.append(li.select('article.product_pod')[0].select_one('h3 a').get('title'))
        links.append( 'https://books.toscrape.com/'+ li.select('article.product_pod')[0].select_one('h3 a').get('href') )


# 시각화======================
data = {
    'title' : titles,
    'links' : links
}
pd.DataFrame(data)

,title,links
0,A Light in the Attic,https://books.toscrape.com/a-light-in-the-atti...
1,Tipping the Velvet,https://books.toscrape.com/tipping-the-velvet_...
2,Soumission,https://books.toscrape.com/soumission_998/inde...
3,Sharp Objects,https://books.toscrape.com/sharp-objects_997/i...
4,Sapiens: A Brief History of Humankind,https://books.toscrape.com/sapiens-a-brief-his...
...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,https://books.toscrape.com/alice-in-wonderland...
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",https://books.toscrape.com/ajin-demi-human-vol...
997,A Spy's Devotion (The Regency Spies of London #1),https://books.toscrape.com/a-spys-devotion-the...
998,1st to Die (Women's Murder Club #1),https://books.toscrape.com/1st-to-die-womens-m...


## 실습
https://news.naver.com/section/100

In [48]:
import requests
from bs4 import BeautifulSoup
url = 'https://news.naver.com/section/100'

response = requests.get(url)

soup = BeautifulSoup(response.text, 'html.parser')

result = soup.select_one('div.section_article.as_headline._TEMPLATE')
for data in result.select('ul>li'):
    print(data.select_one('strong.sa_text_strong').text)

군, 李 정부 출범 후 첫 천안함 추모 행사…국방 차관 대표로 참석
박수민 의원 “주택·교통·출산 ‘연쇄 악순환’ 정면 돌파하겠다”…서울시장 출마 공식 선언
佛 라팔 뺨치는 4.5세대 초음속기…가성비에 亞·유럽 '러브콜'
한동훈 "민주당, 도망 다니지 말라… 그 많은 의석수 부끄러워"
KB국민은행, 안중근 의사 순국일 특별영상 공개
주한 이란대사 “韓, 참혹한 사태와 실패 공범되지 않길”
[속보] 이 대통령 "중동사태 예측 어려워… 국민 에너지 절약 적극 동참해 달라"
이 대통령 지지율 69% '역대 최고치'…민주 46% vs 국힘 18%
“성범죄자·마약범까지 군으로”…군 병력 감축 속 ‘신원특이자’ 3년 새 50% 급증
"행정수도특별법 더 미룰 수 없다"…여야 의원들 조속 처리 촉구
